In [1]:
# 01 — Data Acquisition with Google Earth Engine

#This notebook inspects and exports remotely sensed inputs for satellite-based flood extent and damage-proxy analysis in Nepal.

#**Phase:** 1 — Data acquisition  
#**Status:** Exploratory / event window under verification

In [1]:
from pathlib import Path
import sys

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Could not find the repository root containing the 'src' directory. "
        f"Current working directory: {Path.cwd()}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    AOI_BBOX,
    BASELINE_START,
    BASELINE_END,
    FLOOD_START,
    FLOOD_END,
    S1_POLARIZATIONS,
    S1_ORBIT_PASS,
    S2_CLOUD_COVER_MAX,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

Project root: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage
Python executable: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage/.venv/bin/python


In [3]:
EE_PROJECT_ID = "nepal-flood-sentinel"

try:
    ee.Initialize(project=EE_PROJECT_ID)
    print(f"Google Earth Engine initialized successfully: {EE_PROJECT_ID}")
except Exception as error:
    print("Earth Engine initialization failed.")
    print(error)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Google Earth Engine initialized successfully: nepal-flood-sentinel


In [4]:
aoi = ee.Geometry.Rectangle([
    AOI_BBOX["min_lon"],
    AOI_BBOX["min_lat"],
    AOI_BBOX["max_lon"],
    AOI_BBOX["max_lat"],
])

aoi.getInfo()

{'type': 'Polygon',
 'coordinates': [[[85.8, 27.6],
   [86.2, 27.6],
   [86.2, 28.2],
   [85.8, 28.2],
   [85.8, 27.6]]]}

In [5]:
Map = geemap.Map(center=[27.9, 86.0], zoom=9)
Map.addLayer(
    aoi,
    {"color": "yellow"},
    "Initial AOI",
)
Map

Map(center=[27.9, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [6]:
def get_sentinel1_collection(start_date, end_date, orbit_pass=None):
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )

    if orbit_pass is not None:
        collection = collection.filter(
            ee.Filter.eq("orbitProperties_pass", orbit_pass)
        )

    return collection

In [7]:
s1_baseline_desc = get_sentinel1_collection(
    BASELINE_START,
    BASELINE_END,
    S1_ORBIT_PASS,
)

s1_post_desc = get_sentinel1_collection(
    FLOOD_START,
    FLOOD_END,
    S1_ORBIT_PASS,
)

print("Baseline Sentinel-1 scenes, descending:", s1_baseline_desc.size().getInfo())
print("Post-event Sentinel-1 scenes, descending:", s1_post_desc.size().getInfo())

Baseline Sentinel-1 scenes, descending: 11
Post-event Sentinel-1 scenes, descending: 3


In [8]:
def collection_metadata(collection, properties):
    features = collection.map(
        lambda image: ee.Feature(
            None,
            image.toDictionary(properties),
        )
    )
    return pd.DataFrame(features.aggregate_array("system:time_start").getInfo())

s1_properties = [
    "system:time_start",
    "system:index",
    "orbitProperties_pass",
    "relativeOrbitNumber_start",
    "platform_number",
]

def sentinel1_dataframe(collection):
    features = collection.map(
        lambda image: ee.Feature(None, image.toDictionary(s1_properties))
    )
    records = features.getInfo()["features"]
    dataframe = pd.DataFrame(
        [item["properties"] for item in records]
    )
    dataframe["acquisition_time"] = pd.to_datetime(
        dataframe["system:time_start"],
        unit="ms",
        utc=True,
    )
    return dataframe.sort_values("acquisition_time")

s1_baseline_df = sentinel1_dataframe(s1_baseline_all)
s1_post_df = sentinel1_dataframe(s1_post_all)

display(s1_baseline_df)
display(s1_post_df)

NameError: name 's1_baseline_all' is not defined